# ================================================================
# SUMMARISATION EXPERIMENT (NON-RAG) — FEW-SHOT PROMPTING
# ================================================================

## Objective
This experiment evaluates the ability of a Large Language Model (LLM) to generate structured summaries of job descriptions using Few-Shot Prompting without Retrieval-Augmented Generation (RAG).

The experiment is designed to measure how well the LLM summarises job descriptions using only the original input text, without external retrieval or additional context.

---

## Dataset
- Source: Djinni Job Descriptions Dataset
- Dataset version:
  - `djinni_with_reference_summaries.csv`
- Sample size: All records with valid reference summaries

### Columns Used
- `Long Description` → Input job description
- `Reference_Summary` → Ground-truth reference summary
- `Position` → Original job title
- `id` → Unique identifier

---

## Experimental Setup

### Summarisation Approach
- Approach Type: Few-Shot Prompting
- Retrieval: Not used
- RAG: Not used
- Vector Database: Not used
- LLM:
  - `llama-3.3-70b-versatile` using Groq

### Generation Parameters
- Temperature: `0.0`
- Max Tokens: `700`

### Embedding Model for Evaluation
- `sentence-transformers/all-MiniLM-L6-v2`

The embedding model is used only for semantic similarity evaluation and not for generating summaries.

---

## Prompting Strategy

A fixed few-shot prompt is used for all job descriptions.

The prompt instructs the LLM to generate a concise structured summary using only information explicitly present in the job description.

The generated summary must contain exactly five sections:

- Role Overview
- Responsibilities
- Skills
- Tools
- Experience

---

## Methodology

The experiment follows these steps:

1. Load the processed Djinni dataset with reference summaries
2. Clean and preprocess job descriptions
3. Remove invalid or duplicate records
4. Keep only rows with valid reference summaries
5. Build a few-shot summarisation prompt for each job description
6. Send the prompt to the LLM
7. Capture the generated structured summary
8. Compare the generated summary with the reference summary
9. Compute automatic evaluation metrics
10. Save results and final summary metrics

---

## Evaluation Metrics

### Core Metrics
- ROUGE-L Precision
- ROUGE-L Recall
- ROUGE-L F1
- Semantic Similarity
- Compression Ratio

### Skill Preservation Metrics
- IRR (Information Retention Ratio)
- Hallucination Rate
- Top-K Skill Coverage
- Global Skill Overlap
- Hallucination Prevalence
- Hallucination Density

### Output Format Validation
The notebook also checks whether the generated summary contains all required sections:

- Role Overview
- Responsibilities
- Skills
- Tools
- Experience

---

## Outputs

The experiment generates the following output files:

### Intermediate Outputs
- `results.csv`
- `results.json`
- `failed_results.csv`

### Final Outputs
- `final_metrics.csv`
- `final_metrics.json`


---

## Purpose

This experiment serves as the baseline non-RAG summarisation experiment using Few-Shot Prompting.

In [ ]:
# ================================================================
# INSTALL REQUIRED LIBRARIES
# ================================================================
# ================================================================
# INSTALL REQUIRED LIBRARIES (RUN ONCE PER SESSION)
# ================================================================
!pip install -q pandas numpy rouge-score sentence-transformers scikit-learn groq tenacity tqdm
print("=" * 60)
print("Required Libraries installed")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.3 MB/s eta 0:00:00
Required Libraries installed


In [ ]:
# ================================================================
# 1. BASIC PYTHON LIBRARIES
# ================================================================
import os
import re
import json
import warnings
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ================================================================
# 2. NLP / EVALUATION LIBRARIES
# ================================================================
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ================================================================
# 3. LLM LIBRARIES
# ================================================================
from groq import Groq, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# ================================================================
# 4. GOOGLE COLAB / DRIVE
# ================================================================
from google.colab import drive, userdata

# ================================================================
# 5. VERSION CHECKS
# ================================================================
def get_version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return "installed"

print("=" * 60)
print("LIBRARY VERSION CHECK")
print("=" * 60)
print("pandas               :", pd.__version__)
print("numpy                :", np.__version__)
print("rouge-score          :", get_version("rouge-score"))
print("sentence-transformers:", get_version("sentence-transformers"))
print("groq                 :", get_version("groq"))
print("=" * 60)

# ================================================================
# 6. MOUNT GOOGLE DRIVE
# ================================================================
drive.mount('/content/drive')

# Root project directory in Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Masters/Masters_Thesis_Project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

# ================================================================
# 7. LOAD GROQ API KEY
# ================================================================
# Make sure GROQ_API_KEY exists in Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY_1")
assert GROQ_API_KEY, "GROQ_API_KEY is missing in Colab Secrets."

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("GROQ API key loaded from Colab secrets")

# ================================================================
# 8. EXPERIMENT CONFIGURATION
# ================================================================
# File paths
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "reference_summaries"
REFERENCE_DATASET_PATH = PROCESSED_DATA_DIR / "djinni_with_reference_summaries.csv"

# Column names
TEXT_COLUMN = "Long Description"
ID_COLUMN = "id"
POSITION_COLUMN = "Position"
REFERENCE_SUMMARY_COLUMN = "Reference_Summary"

# Experiment settings
EXPERIMENT_ID = "exp_s_05_few_shot_without_rag_llama-3.3-70b-versatile"
PROMPT_TYPE = "few_shot"
MODEL_NAME = "llama-3.3-70b-versatile"
TEMPERATURE = 0.0
MAX_TOKENS = 700

# Embedding model for semantic similarity
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Output paths
OUTPUT_DIR = PROJECT_DIR / "outputs" / "experiments" / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_RESULTS_JSON = OUTPUT_DIR / f"{EXPERIMENT_ID}.json"
BATCH_RESULTS_CSV = OUTPUT_DIR / f"{EXPERIMENT_ID}.csv"
FAILED_RESULTS_CSV = OUTPUT_DIR / f"{EXPERIMENT_ID}_failed.csv"

print("Experiment ID :", EXPERIMENT_ID)
print("Model         :", MODEL_NAME)
print("Temperature   :", TEMPERATURE)
print("Max tokens    :", MAX_TOKENS)
print("Reference CSV :", REFERENCE_DATASET_PATH)
print("Output dir    :", OUTPUT_DIR)

# ================================================================
# 9. LOAD DATASET FROM CSV WITH REFERENCE SUMMARIES
# ================================================================
# IMPORTANT:
# We are NOT loading the raw Hugging Face dataset here.
# We are loading the processed CSV that already contains Reference_Summary.
print("=" * 60)
print("Loading processed dataset with reference summaries...")

df = pd.read_csv(REFERENCE_DATASET_PATH)

print("=" * 60)
print("Dataset loaded successfully")
print("Shape   :", df.shape)
print("Columns :", list(df.columns))

display(df.head(3))
print("=" * 60)

# ================================================================
# 10. DATA CLEANING / PREPROCESSING
# ================================================================
def clean_text(text: str) -> str:
    """
    Clean noisy job description text.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove HTML tags if any
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs if any
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def preprocess_dataset(df: pd.DataFrame, text_column: str, id_column: str) -> pd.DataFrame:
    """
    Remove null, duplicate, and noisy records.
    """
    work_df = df.copy()

    # Keep only rows with valid text and id
    work_df = work_df[work_df[text_column].notna()]
    work_df = work_df[work_df[id_column].notna()]

    # Clean text
    work_df[text_column] = work_df[text_column].apply(clean_text)

    # Remove empty records after cleaning
    work_df = work_df[work_df[text_column].str.strip().ne("")]

    # Remove duplicate Long Description entries
    work_df = work_df.drop_duplicates(subset=[text_column]).reset_index(drop=True)

    return work_df


clean_df = preprocess_dataset(
    df=df,
    text_column=TEXT_COLUMN,
    id_column=ID_COLUMN
)

print("Original shape :", df.shape)
print("Cleaned shape  :", clean_df.shape)

display(clean_df[[ID_COLUMN, POSITION_COLUMN, TEXT_COLUMN]].head(3))

# ================================================================
# 11. FILTER ROWS WITH REFERENCE SUMMARIES
# ================================================================
if REFERENCE_SUMMARY_COLUMN not in clean_df.columns:
    raise ValueError(f"{REFERENCE_SUMMARY_COLUMN} column not found in dataset.")

# Standardize Reference_Summary column
clean_df[REFERENCE_SUMMARY_COLUMN] = clean_df[REFERENCE_SUMMARY_COLUMN].fillna("").astype(str)

# Convert literal 'nan' strings to empty
clean_df.loc[
    clean_df[REFERENCE_SUMMARY_COLUMN].str.strip().str.lower() == "nan",
    REFERENCE_SUMMARY_COLUMN
] = ""

# Keep only rows that already have reference summaries
valid_df = clean_df[
    clean_df[REFERENCE_SUMMARY_COLUMN].str.strip().ne("")
].copy()

print("Rows with reference summaries:", len(valid_df))

# Run on 10 samples first
experiment_df = valid_df.copy().reset_index(drop=True)

print("Experiment rows selected:", len(experiment_df))
print("=" * 60)
display(experiment_df[[ID_COLUMN, POSITION_COLUMN]].head(10))

# ================================================================
# 12. PROMPT FUNCTION
# ================================================================
def build_few_shot_prompt(job_description: str) -> str:
    return f"""
You are an expert HR analyst.

Below are examples of how to summarize job descriptions.

Example 1:

Job Description:
We are looking for a backend developer with experience in Java, Spring Boot, and Microservices. The candidate should work on REST APIs and collaborate with cross-functional teams. Knowledge of Docker and AWS is preferred. Minimum 3 years of experience required.

Summary:

Role Overview:
Backend developer responsible for building scalable APIs.

Responsibilities:
Develop REST APIs and collaborate with teams.

Skills:
Java, Spring Boot, Microservices

Tools:
Docker, AWS

Experience:
3+ years


Example 2:

Job Description:
Seeking a data analyst skilled in Python, SQL, and Tableau. The role involves data cleaning, visualization, and reporting insights to stakeholders. Experience in Excel and communication skills are required.

Summary:

Role Overview:
Data analyst focused on deriving insights from data.

Responsibilities:
Analyze data and create reports for stakeholders.

Skills:
Python, SQL, Tableau

Tools:
Excel

Experience:
Not explicitly stated


Now summarize the following job description in the SAME format.

Rules:
- Use only information from the text
- Do not add new information
- Keep it concise
- Skills and Tools must be comma-separated
- If missing, write: Not explicitly stated
- Do not include reasoning

Job Description:
{job_description}
""".strip()


# ================================================================
# 13. GROQ CLIENT
# ================================================================
class LLMClientError(Exception):
    """
    Custom error for LLM failures.
    """
    pass


class GroqClient:
    """
    Groq client wrapper for summarisation experiments.
    """

    def __init__(self, model_name: str):
        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY is missing from environment variables.")
        self.client = Groq(api_key=api_key)
        self.model_name = model_name

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type(Exception),
        reraise=True
    )
    def generate(
        self,
        prompt: str,
        temperature: float,
        max_tokens: int
    ) -> str:
        completion = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": "You are a precise and factual HR summarisation assistant."},
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )

        if not completion.choices:
            raise LLMClientError("Groq returned no choices.")

        text = completion.choices[0].message.content
        if not text or not str(text).strip():
            raise LLMClientError("Groq returned an empty response.")

        return str(text).strip()


groq_client = GroqClient(model_name=MODEL_NAME)
print("Groq client initialized")


# ================================================================
# 14. EVALUATION FUNCTIONS
# ================================================================
_embedding_model_cache = {}

def get_embedding_model(model_name: str):
    """
    Cache embedding model so it loads only once.
    """
    if model_name not in _embedding_model_cache:
        _embedding_model_cache[model_name] = SentenceTransformer(model_name)
    return _embedding_model_cache[model_name]


def compute_rouge_l(reference_text: str, generated_text: str):
    """
    Compute ROUGE-L precision, recall, and F1.
    """
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    score = scorer.score(reference_text, generated_text)["rougeL"]
    return {
        "rougeL_precision": round(score.precision, 4),
        "rougeL_recall": round(score.recall, 4),
        "rougeL_f1": round(score.fmeasure, 4),
    }


def compute_semantic_similarity(reference_text: str, generated_text: str, embedding_model_name: str):
    """
    Compute semantic similarity using sentence embeddings.
    """
    model = get_embedding_model(embedding_model_name)
    embeddings = model.encode([reference_text, generated_text], convert_to_numpy=True)
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return round(float(similarity), 4)


def compute_compression_ratio(original_text: str, summary_text: str):
    """
    Compression ratio = summary length / original JD length
    """
    original_length = max(len(original_text.split()), 1)
    summary_length = len(summary_text.split())
    return round(summary_length / original_length, 4)


def validate_experiment_summary_sections(summary_text: str):
    """
    Check whether required output sections are present.
    """
    required_sections = [
        "Role Overview",
        "Responsibilities",
        "Skills",
        "Tools",
        "Experience",
    ]
    lowered = str(summary_text).lower()
    return {
        section: f"{section.lower()}:" in lowered
        for section in required_sections
    }


# ================================================================
# 15. SKILL EXTRACTION AND CUSTOM METRICS
# ================================================================
def normalize_skill(text: str) -> str:
    """
    Normalize extracted skill phrase.
    """
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^[,;:\-\.\s]+|[,;:\-\.\s]+$", "", text)
    return text


def extract_skills(text: str):
    """
    Extract explicitly mentioned skills without using a predefined skill lexicon.

    Strategy:
    1. Capture phrases after common skill-introducing patterns
    2. Split multi-skill phrases by commas / and / slash
    3. Capture likely capitalized technical terms
    4. Normalize and deduplicate
    """
    if not text or not isinstance(text, str):
        return []

    text = re.sub(r"\s+", " ", text).strip()
    extracted = []

    patterns = [
        r"(?:experience with|experience in|proficient in|proficiency in|knowledge of|familiarity with|expertise in|skills in|skilled in)\s+([A-Za-z0-9\+\#\./,\-\s]+?)(?:\.|;|:|, and| and |\n|$)",
        r"(?:tools?:|technologies:|skills:)\s*([A-Za-z0-9\+\#\./,\-\s]+?)(?:\.|\n|$)",
        r"(?:requirements?:|required skills:)\s*([A-Za-z0-9\+\#\./,\-\s]+?)(?:\.|\n|$)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        for match in matches:
            parts = re.split(r",| and |/|\|", match)
            for part in parts:
                skill = normalize_skill(part)
                if skill and len(skill) > 1:
                    extracted.append(skill)

    tech_terms = re.findall(
        r"\b(?:[A-Z][A-Za-z0-9\+\#\.]{1,}|[A-Za-z]+\+[A-Za-z]+|[A-Za-z]+#[A-Za-z]+)\b",
        text
    )
    for term in tech_terms:
        skill = normalize_skill(term)
        if skill and len(skill) > 1:
            extracted.append(skill)

    blacklist = {
        "experience",
        "skills",
        "tools",
        "technologies",
        "knowledge",
        "familiarity",
        "expertise",
        "requirements",
        "required skills",
    }

    extracted = [s for s in extracted if s not in blacklist]

    return sorted(list(set(extracted)))


def build_skill_analysis(original_text: str, summary_text: str):
    """
    Extract skills from original JD and generated summary.
    Then compute IRR and hallucinated skills.
    """
    jd_skills = extract_skills(original_text)
    summary_skills = extract_skills(summary_text)

    jd_set = set(map(normalize_skill, jd_skills))
    summary_set = set(map(normalize_skill, summary_skills))

    irr = len(jd_set & summary_set) / len(jd_set) if len(jd_set) > 0 else 0.0
    hallucinated_skills = sorted(list(summary_set - jd_set))

    return {
        "jd_skills": sorted(list(jd_set)),
        "summary_skills": sorted(list(summary_set)),
        "irr": round(irr, 4),
        "hallucinated_skills": hallucinated_skills,
    }


def compute_hallucination_rate(original_skills, summary_skills):
    """
    Hallucination rate = hallucinated skills / total summary skills
    """
    original_set = set(map(normalize_skill, original_skills))
    summary_set = set(map(normalize_skill, summary_skills))

    if len(summary_set) == 0:
        return 0.0

    hallucinated = summary_set - original_set
    return round(len(hallucinated) / len(summary_set), 4)


def compute_top_k_skill_coverage(original_skills, summary_skills, k=5):
    """
    Top-K skill coverage = overlap of first K JD skills with summary skills.
    """
    original_top_k = list(original_skills)[:k]
    original_set = set(map(normalize_skill, original_top_k))
    summary_set = set(map(normalize_skill, summary_skills))

    if len(original_set) == 0:
        return 0.0

    return round(len(original_set & summary_set) / len(original_set), 4)




# ================================================================
# 16. CHECKPOINT / RESUME
# ================================================================
if BATCH_RESULTS_JSON.exists():
    with open(BATCH_RESULTS_JSON, "r", encoding="utf-8") as f:
        loaded = json.load(f)

    if isinstance(loaded, list):
        all_results = loaded
        completed_ids = {str(r["selected_id"]) for r in all_results if "selected_id" in r}
    else:
        all_results = []
        completed_ids = set()
else:
    all_results = []
    completed_ids = set()

failed_results = []
if FAILED_RESULTS_CSV.exists():
    try:
        failed_results = pd.read_csv(FAILED_RESULTS_CSV).to_dict(orient="records")
    except Exception:
        failed_results = []

print("Already completed:", len(completed_ids))
print("Already failed   :", len(failed_results))

for r in all_results:
    if "prompt_type" not in r:
        r["prompt_type"] = PROMPT_TYPE


# ================================================================
# 17. CHECKPOINT SAVE FUNCTION
# ================================================================
def save_checkpoints(all_results, failed_results):
    """
    Save current successful results and failed records.
    """
    with open(BATCH_RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    if all_results:
        checkpoint_df = pd.DataFrame(all_results).copy()

        for col in ["jd_skills", "summary_skills", "hallucinated_skills", "section_validation"]:
            if col in checkpoint_df.columns:
                checkpoint_df[col] = checkpoint_df[col].apply(
                    lambda x: json.dumps(x, ensure_ascii=False)
                )

        checkpoint_df.to_csv(BATCH_RESULTS_CSV, index=False, encoding="utf-8")

    if failed_results:
        pd.DataFrame(failed_results).to_csv(FAILED_RESULTS_CSV, index=False, encoding="utf-8")




# ================================================================
# 18. RUN EXPERIMENT ON 100 RECORDS
# ================================================================
for i, row in tqdm(
    experiment_df.iterrows(),
    total=len(experiment_df),
    desc="Running Summarization Experiment",
    unit="JD"
):
    current_id = str(row[ID_COLUMN])

    if current_id in completed_ids:
        tqdm.write(f"Skipped {i+1}/{len(experiment_df)} | ID: {current_id}")
        continue

    tqdm.write(f"Running {i+1}/{len(experiment_df)} | ID: {current_id}")

    try:
        job_description = row[TEXT_COLUMN]
        reference_summary = row[REFERENCE_SUMMARY_COLUMN]

        prompt = build_few_shot_prompt(job_description=job_description)

        generated_summary = groq_client.generate(
            prompt=prompt,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        rouge_scores = compute_rouge_l(reference_summary, generated_summary)

        semantic_similarity = compute_semantic_similarity(
            reference_text=reference_summary,
            generated_text=generated_summary,
            embedding_model_name=EMBEDDING_MODEL_NAME
        )

        compression_ratio = compute_compression_ratio(
            original_text=job_description,
            summary_text=generated_summary
        )

        skill_analysis = build_skill_analysis(
            original_text=job_description,
            summary_text=generated_summary
        )

        hallucination_rate = compute_hallucination_rate(
            original_skills=skill_analysis["jd_skills"],
            summary_skills=skill_analysis["summary_skills"]
        )

        top_k_skill_coverage = compute_top_k_skill_coverage(
            original_skills=skill_analysis["jd_skills"],
            summary_skills=skill_analysis["summary_skills"],
            k=5
        )

        section_validation = validate_experiment_summary_sections(
            generated_summary
        )

        result = {
            "experiment_id": EXPERIMENT_ID,
            "selected_id": current_id,
            "prompt_type": PROMPT_TYPE,
            "model_name": MODEL_NAME,
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS,
            "job_description": job_description,
            "reference_summary": reference_summary,
            "generated_summary": generated_summary,
            **rouge_scores,
            "semantic_similarity": semantic_similarity,
            "compression_ratio": compression_ratio,
            "jd_skills": skill_analysis["jd_skills"],
            "summary_skills": skill_analysis["summary_skills"],
            "irr": skill_analysis["irr"],
            "hallucinated_skills": skill_analysis["hallucinated_skills"],
            "hallucination_rate": hallucination_rate,
            "top_k_skill_coverage": top_k_skill_coverage,
            "section_validation": section_validation,
        }

        all_results.append(result)
        completed_ids.add(current_id)

        save_checkpoints(all_results, failed_results)

        tqdm.write(
            f"Completed {i+1}/{len(experiment_df)} | "
            f"ID: {current_id} | "
            f"ROUGE-L: {result['rougeL_f1']} | "
            f"Sim: {result['semantic_similarity']} | "
            f"IRR: {result['irr']}"
        )

    except RateLimitError as e:
        tqdm.write("Rate limit hit. Checkpoint saved.")
        tqdm.write(str(e))
        save_checkpoints(all_results, failed_results)
        break

    except Exception as e:
        tqdm.write(
            f"Failed {i+1}/{len(experiment_df)} | "
            f"ID: {current_id} | Error: {e}"
        )

        failed_results.append({
            "selected_id": current_id,
            "error": str(e)
        })

        save_checkpoints(all_results, failed_results)
        continue



# ================================================================
# 19. FINAL SUMMARY METRICS AND SAVE SEPARATE CSV
# ================================================================
results_df = pd.DataFrame(all_results)

FINAL_METRICS_CSV = OUTPUT_DIR / f"{EXPERIMENT_ID}_final_metrics.csv"
FINAL_METRICS_JSON = OUTPUT_DIR / f"{EXPERIMENT_ID}_final_metrics.json"

print("Saved JSON to:", BATCH_RESULTS_JSON)
print("Saved CSV to :", BATCH_RESULTS_CSV)

METRIC_COLUMNS = [
    "rougeL_f1",
    "semantic_similarity",
    "irr",
    "hallucination_rate",
    "compression_ratio",
    "top_k_skill_coverage",
]

def compute_global_skill_overlap(results_df):
    all_jd_skills = set()
    all_summary_skills = set()

    for _, row in results_df.iterrows():
        jd_skills = row["jd_skills"]
        summary_skills = row["summary_skills"]

        if isinstance(jd_skills, list):
            all_jd_skills.update(jd_skills)

        if isinstance(summary_skills, list):
            all_summary_skills.update(summary_skills)

    if len(all_jd_skills) == 0:
        return 0.0

    overlap = all_jd_skills & all_summary_skills
    return round(len(overlap) / len(all_jd_skills), 4)


def compute_hallucination_prevalence(results_df):
    hallucinated_count = results_df["hallucinated_skills"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    summaries_with_hallucination = (hallucinated_count > 0).sum()
    total_summaries = len(results_df)

    prevalence = summaries_with_hallucination / total_summaries if total_summaries > 0 else 0.0

    affected = hallucinated_count[hallucinated_count > 0]
    density = affected.mean() if len(affected) > 0 else 0.0

    return round(prevalence, 4), round(density, 4)


if not results_df.empty:
    final_metrics = {
        "experiment_id": EXPERIMENT_ID,
        "prompt_type": PROMPT_TYPE,
        "model_name": MODEL_NAME,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "num_samples_completed": len(results_df),
    }

    for metric in METRIC_COLUMNS:
        final_metrics[f"{metric}_mean"] = round(results_df[metric].mean(), 4)
        final_metrics[f"{metric}_median"] = round(results_df[metric].median(), 4)
        final_metrics[f"{metric}_std"] = round(results_df[metric].std(), 4)
        final_metrics[f"{metric}_min"] = round(results_df[metric].min(), 4)
        final_metrics[f"{metric}_max"] = round(results_df[metric].max(), 4)

    global_skill_overlap = compute_global_skill_overlap(results_df)
    hallucination_prevalence, hallucination_density = compute_hallucination_prevalence(results_df)

    final_metrics["global_skill_overlap"] = global_skill_overlap
    final_metrics["hallucination_prevalence"] = hallucination_prevalence
    final_metrics["hallucination_density"] = hallucination_density

    final_metrics_df = pd.DataFrame([final_metrics])
    final_metrics_df.to_csv(FINAL_METRICS_CSV, index=False, encoding="utf-8")

    print("\n===== FINAL SUMMARY METRICS =====")
    print("ROUGE-L Mean:", final_metrics["rougeL_f1_mean"])
    print("ROUGE-L Median:", final_metrics["rougeL_f1_median"])
    print("ROUGE-L Std:", final_metrics["rougeL_f1_std"])

    print("Semantic Similarity Mean:", final_metrics["semantic_similarity_mean"])
    print("IRR Mean:", final_metrics["irr_mean"])
    print("Global Skill Overlap:", final_metrics["global_skill_overlap"])

    print("Hallucination Rate Mean:", final_metrics["hallucination_rate_mean"])
    print("Hallucination Prevalence:", final_metrics["hallucination_prevalence"])
    print("Hallucination Density:", final_metrics["hallucination_density"])

    print("Compression Ratio Mean:", final_metrics["compression_ratio_mean"])
    print("Top-K Skill Coverage Mean:", final_metrics["top_k_skill_coverage_mean"])

    print("\nSaved final metrics to:", FINAL_METRICS_CSV)

    # Save JSON
    with open(FINAL_METRICS_JSON, "w", encoding="utf-8") as f:
      json.dump(final_metrics, f, indent=4, ensure_ascii=False)

    print("Saved final metrics JSON to:", FINAL_METRICS_JSON)

    display(final_metrics_df)
else:
    print("No results available yet.")

LIBRARY VERSION CHECK
pandas               : 2.2.2
numpy                : 2.0.2
rouge-score          : 0.1.2
sentence-transformers: 5.4.0
groq                 : 1.2.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/Colab Notebooks/Masters/Masters_Thesis_Project
GROQ API key loaded from Colab secrets
Experiment ID : exp_s_05_few_shot_without_rag_llama-3.3-70b-versatile
Model         : llama-3.3-70b-versatile
Temperature   : 0.0
Max tokens    : 700
Reference CSV : /content/drive/MyDrive/Colab Notebooks/Masters/Masters_Thesis_Project/data/reference_summaries/djinni_with_reference_summaries.csv
Output dir    : /content/drive/MyDrive/Colab Notebooks/Masters/Masters_Thesis_Project/outputs/experiments/exp_s_05_few_shot_without_rag_llama-3.3-70b-versatile
Loading processed dataset with reference summaries...
Dataset loaded successfully
Shape   : (100, 11)
Columns : ['Posit

,Position,Long Description,Company Name,Exp Years,Primary Keyword,English Level,Published,Long Description_lang,id,index_level_0,Reference_Summary
0,Java Developer,The job holder will . Design develop and maint...,Quickstarter,3y,Java,upper,2021-12-01T00:00:00+02:00,en,67aefec1-1efc-5c5b-b2db-c0ef5b74f519,77261,Role Overview:\nThe Java Developer will design...
1,Integration specialist / DevOps,Project Description Join our Development Cente...,Luxoft,3y,DevOps,upper,2022-04-01T00:00:00+03:00,en,2ec599d6-1509-5c34-9aec-54ac2e04b2d5,72279,Role Overview:\nThe Integration Specialist / D...
2,Head of Marketing,TurnKey Labs is the most trusted offshore soft...,TurnKey Labs,5y,Marketing,upper,2021-10-01T00:00:00+03:00,en,edba8bfd-8a91-5b37-9f72-bde92397e804,69706,"Role Overview:\nReporting to the CEO, the Head..."


Original shape : (100, 11)
Cleaned shape  : (100, 11)


,id,Position,Long Description
0,67aefec1-1efc-5c5b-b2db-c0ef5b74f519,Java Developer,The job holder will . Design develop and maint...
1,2ec599d6-1509-5c34-9aec-54ac2e04b2d5,Integration specialist / DevOps,Project Description Join our Development Cente...
2,edba8bfd-8a91-5b37-9f72-bde92397e804,Head of Marketing,TurnKey Labs is the most trusted offshore soft...


Rows with reference summaries: 100
Experiment rows selected: 100


...

Running 1/100 | ID: 67aefec1-1efc-5c5b-b2db-c0ef5b74f519


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Completed 1/100 | ID: 67aefec1-1efc-5c5b-b2db-c0ef5b74f519 | ROUGE-L: 0.3947 | Sim: 0.7796 | IRR: 0.3214
Running 2/100 | ID: 2ec599d6-1509-5c34-9aec-54ac2e04b2d5
Completed 2/100 | ID: 2ec599d6-1509-5c34-9aec-54ac2e04b2d5 | ROUGE-L: 0.4955 | Sim: 0.8687 | IRR: 0.4286
Running 3/100 | ID: edba8bfd-8a91-5b37-9f72-bde92397e804
Completed 3/100 | ID: edba8bfd-8a91-5b37-9f72-bde92397e804 | ROUGE-L: 0.3365 | Sim: 0.848 | IRR: 0.0448
Running 4/100 | ID: 077025bf-2c32-5855-9211-68b09228a635
Completed 4/100 | ID: 077025bf-2c32-5855-9211-68b09228a635 | ROUGE-L: 0.4148 | Sim: 0.7322 | IRR: 0.2128
Running 5/100 | ID: adb62bd6-998e-58ec-a32c-3b99fe814b04
Completed 5/100 | ID: adb62bd6-998e-58ec-a32c-3b99fe814b04 | ROUGE-L: 0.4513 | Sim: 0.6742 | IRR: 0.3
Running 6/100 | ID: b2315b02-ac50-551c-93b5-a88d6ee230e6
Completed 6/100 | ID: b2315b02-ac50-551c-93b5-a88d6ee230e6 | ROUGE-L: 0.4865 | Sim: 0.8787 | IRR: 0.4884
Running 7/100 | ID: 881421a8-07a9-59a0-9765-6c48609ae879
Completed 7/100 | ID: 881421a8-0

,experiment_id,prompt_type,model_name,temperature,max_tokens,num_samples_completed,rougeL_f1_mean,rougeL_f1_median,rougeL_f1_std,rougeL_f1_min,...,compression_ratio_min,compression_ratio_max,top_k_skill_coverage_mean,top_k_skill_coverage_median,top_k_skill_coverage_std,top_k_skill_coverage_min,top_k_skill_coverage_max,global_skill_overlap,hallucination_prevalence,hallucination_density
0,exp_s_05_few_shot_without_rag_llama-3.3-70b-ve...,few_shot,llama-3.3-70b-versatile,0.0,700,100,0.3635,0.3598,0.0644,0.1858,...,0.1007,0.7671,0.31,0.4,0.2077,0.0,0.8,0.3299,1.0,4.97
